## LArPix-v3c Verification using RTLSwarm 

Kalindi Gosine 2026-07-17 

This document outlines a selection of the scenarios for LArPix networks of varying sizes to verify chip-level and network-level behavior of the v3c RTL. All simulations are done using RTLSwarm, a Verilator-based distributed RTL simulation framework. 

## Packet Loss 2x2: Continuous Data Packet Receiving

This scenario launches a 2x2 network of LArPix chips and injects those with chip_id 1 and 2 with *-5e-15* charge into the analog front end on all 64 channels of each chip at tick=1,000. This leads to the formation of 64 data packets in chips 1 and 2, which enter the shared FIFO. These two chips only have one UART lane enabled for downstream transmission (orange arrows), each pointing towards Chip 0, the sink chip. The sink chip is connected to the data collector, which acts as the FPGA analog (purple arrow). The goal of this test is to see how Chip 0 handles receiving continuous data packets along two receiving UART lanes simultaneously and how the Hydra state machine handles routing these data packets out of Chip 0 towards the FPGA (on the south lane of Chip 0). 

The network in this simulation run has been preconfigured, meaning that the configuration register was set before the simulation run began. The relevant configuration register changes set the tx_enable_downstream and tx_enable_upstream bits to control which downstream (orange) and upstream (blue) transmission lanes are enabled for each chip, as well as setting the chip_id for each chip (upon startup, all chip_id=1). Finally, the csa_threshold for each channel on the chip is changed to 

<figure>
  <img src="assets/v3c_packet_loss_network.png" width="800">
  <figcaption><em>Figure 1: Network view of simulation run displaying data packets from chip1 and chip2 arriving simultaneously on the North and East receiving lanes of Chip 0. </em></figcaption>
</figure>

128 data packets were created during the simulation due to the analog charge injection on chips 1 and 2. By the end of the simulation 128 unique packets reached the FPGA for 100% data packet recovery. 

<figure>
  <img src="assets/packet_loss_debug.png" width="900">
  <figcaption><em>Figure 2: State-level view of Chip0 during the simulation. The bottom timeline shows how the chip handles unloading data from each receiving UART sequentially and loading it into the shared FIFO, as well as Hydra_ctrl cycling through its state machine to receive and transmit packets. The hydra_state signal decodes as IDLE=0, TX_GET_FIFO=TG, TX_WAIT_FIFO=TW, TX_SEND=TS, RX_CAPTURE-RC, RX_PROCESS=RP. </em></figcaption>
</figure>

Data packets arriving on the North and East lanes are parallelized by the receiving UART which, upon completion, holds the packet and asserts !rx_empty. As each lane's empty signal goes low, the hydra_state changes from IDLE to RX_CAPTURE, selecting first the North lane's data packet and then the East lane's data packet. We see that the north_rx_empty signal goes high (so the packet was unloaded from the UART) before the east_rx_empty signal, also confirming that the North UART was selected by Hydra for unloading before the East UART. 

In the bottom timeline, we see that the shared FIFO is not empty and the South lane, which is enabled for downstream transmission, is passing the received packets to the FPGA downstream. When the South TX lane becomes non-busy, and Hydra is in IDLE, the TX_GET_FIFO, TX_WAIT_FIFO, and TX_SEND_FIFO cycle begins to launch a new packet from the FIFO to the South TX lane. 

## Packet Loss 3x3: Data Packets + Pass-Thru Configuration Packets

This scenario launches a 3x3 network of chips where chips 3, 5, and 7 have only one downstream TX lane abled, pointing towards chip 4. Therefore, chip 4 is receiving data packets continuously along its North, East, and West lanes. Additionally, during this period of high data packet movement through the network, the FPGA launches four configuration packets which are register reads of chip 7. For chip 4, these packets are pass-through configuration packets since their target id does not match the current chip id. Therefore, these packets are treated as upstream pass through packets and are handled by the Hydra through the TX_UPSTREAM state. These packets do not enter chip 4's shared FIFO and are instead held in the Hydra until the upstream transmission lane is not busy and able to launch the packet. This presents an additional challenge for the system as Hydra handles data packets arriving on three lanes and pass through configuration packets arriving on its fourth lane simultaneously. 

<figure>
  <img src="assets/higher_stress_network_view.png" width="900">
  <figcaption><em>Figure 3: Network view of scenario with data packets routed to chip 4 along three directions and upstream packets arriving at chip 4 via the south lane. </em></figcaption>
</figure>

Despite the higher level of stress in the system, there is still no packet loss for this scenario with all 192 generated data packets arriving at the FPGA despite all having to converge at chip 4. The configuration read packets targetted at chip 7 also reached chip 7 and the readback replies all entered the chip 7 shared FIFOs and arrived at the FPGA. 

<figure>
  <img src="assets/higher_stress_timeline_view.png" width="900">
  <figcaption><em>Figure 4: State-level view of chip 4 while receiving data and configuration packets. Note how now hydra_state enters state TU=TX_UPSTREAM to transmit the configuration read packets to chip 7. </em></figcaption>
</figure>

## Startup Configuration 10x10

This scenario launches a 10x10 network of LArPix chips without preconfiguration. Therefore, at the start of the runtime, all chips begin with the default register configurations, including chip_id=1. The FPGA sends a sequence of configuration write packets which modify the enabled upstream and downstream transmission lanes as well as the chip_id. The process involves configuring those chips that already had their chip_id reassigned, to have only a single upstream transmission route to a chip which still has its default chip_id=1. Then, a configuration write is sent to change the chip_id from 1 to the desired value. This ensures that although at the start of the simulation all chips have chip_id=1, each chip is momentarily solely accessible via the upstream transmission route and can be uniquely reconfigured via configuration write packets. 

<figure>
  <img src="assets/10x10_assignment.png" width="900">
  <figcaption><em>Figure 5a: Part way through the chip ID assignment process. Note the chips in the bottom lane of chips which have only one upstream (blue) tx lane enabled; therefore, only a single chip with chip_id=1 is reachable in the network. After each chip_id reassignment, configuration writes are sent to modify the routing to reach the next chip for ID reassignment. </em></figcaption>
</figure>

<figure>
  <img src="assets/10x10_completed.png" width="900">
  <figcaption><em>Figure 5b: The state of the network after completion of chip_id and routing assignment. Note how the routing on the bottom lane is modified now so that all chips in the network are accessible via the upstream (blue) tx path.  </em></figcaption>
</figure>

A simulation-related note: the number of ticks/second  has significantly decreased as the number of chip processes launched (100) has exceeded the available CPU cores on the simulation machine. This framework is designed such that it could be run across various machines, utilizing sufficient cores such that simulation time does not scale linearly with network size as it has in this example.  

## Simulated Charge Deposition Tracks

A 15x15 network was launched and charge was injected into channels 0-15 on a total of 13 chips in the network. In a single chip, charge was desposited into the analog front end of the 16 selected channels at the same simulation tick. The bottom track received its first charge injection at tick 1,000 with charge injected into chips separated by 500 ticks. The upper track first received charetg at tick 2,000 with the same 500 tick gap bewteen injections of each chip in the track. This scenario was designed to mimic the kind of spatial data packet formation we could see in a real detector. Future simulations could use real detector data (or detector simulation such as from GEANT4) containing the spatial and temporal distribution of charge deposition scaled to the expected ASIC pixel pitch as an input to simulate the expected data packet output and routing towards a sink chip. 

The charge deposited caused the creation of 208 unique data packets, all of which were recovered at the FPGA. 

<figure>
  <img src="assets/multitrack_partway.png" width="900">
  <figcaption><em>Figure 6a: The state of the network partway through the charge deposition on the chip processes. The chips on the bottom track have begin transmitting data packets which route towards the sink chip.   </em></figcaption>
</figure>

<figure>
  <img src="assets/multitrack_done.png" width="900">
  <figcaption><em>Figure 6b: The state of the network after near the end of the simulation with the very last data packets from the chips in the upper track, which were hit last, travelling towards the sink chip.   </em></figcaption>
</figure>

We can also track the FIFO occupancy of all chips in the network to understand points of congestion. Below are the FIFO occupancy of six selected chips (marked on the left-most figure) over the course of the runtime. We see for chips which experiences analog charge injection and locally generated data packets (chips 64, 96, and 146) that they have a period where their FIFO occupancy increases rapidly as packets enter the FIFO in parallel. Since packets enter and exit chips serially, we see steps of 66 ticks as Hydra waits for the TX UART to become not busy to unload a new packet from the FIFO. 

<figure>
  <div style="display:flex; gap:16px; align-items:flex-start;">
    <img src="assets/selected_chips.png" style="width:35%;">
    <img src="assets/multitrack_fifo.png" style="width:65%;">
  </div>
  <figcaption><em>Figure 8: FIFO occupancy for selected chips in the network across the runtime. </em></figcaption>
</figure>